# TITLE

* ADD DOCUMENTATION WHAT IS THE GOAL/PUROSE of this notebook?

* ADD MARKDOWN TO NOTEBOOK TO OUTLINE AND WALK THROUGH THE STEPS AND MAKE OBSERVATIONS ETC

In [1]:
# the necessary imports
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

In [2]:
oscars = pd.read_csv(
    "https://raw.githubusercontent.com/DLu/oscar_data/main/oscars.csv",
    sep="\t",
    encoding="utf-8"
)
print(oscars.shape)
oscars.head()

(12118, 14)


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins|The Patent Leather Kid,NaN,NaN
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexand...,NaN,NaN
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True,Diane|Angela|The Wife,NaN,NaN
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,NaN,Sadie Thompson,NaN,NaN


In [5]:
# Clean up oscars.csv 
oscars['Winner'] = oscars['Winner'].fillna(False)
def make_slug(title):
    """Convert a film title to The Numbers URL slug."""
    slug = str(title).strip()
    slug = re.sub(r"[''']", "",  slug)       
    slug = re.sub(r"[:&,!?.]", "", slug)     
    slug = re.sub(r"\s+", "-", slug)         
    return slug

oscars['slug'] = oscars['Film'].apply(make_slug)
print(oscars[['Film', 'slug']].head(20).to_string())
oscars_modern = oscars[oscars['Ceremony'] >= 73].copy()

                                               Film                                             slug
0                  The Noose|The Patent Leather Kid                 The-Noose|The-Patent-Leather-Kid
1             The Last Command|The Way of All Flesh            The-Last-Command|The-Way-of-All-Flesh
2                                   A Ship Comes In                                  A-Ship-Comes-In
3                   7th Heaven|Street Angel|Sunrise                  7th-Heaven|Street-Angel|Sunrise
4                                    Sadie Thompson                                   Sadie-Thompson
5                                           Sunrise                                          Sunrise
6                                  The Dove|Tempest                                 The-Dove|Tempest
7                                        7th Heaven                                       7th-Heaven
8   The Devil Dancer|The Magic Flame|Sadie Thompson  The-Devil-Dancer|The-Magic-Flame|Sadie

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (academic research project)"}
BASE_URL = "https://www.the-numbers.com/movie/"
def scrape_film(slug):
    url = f"{BASE_URL}{slug}?public=true"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return None, url, r.status_code
        
        soup = BeautifulSoup(r.text, "html.parser")
        data = {"url": url}
        
        # The Numbers puts financials in a <div id="summary"> table
        summary = soup.find("div", id="summary")
        if summary:
            for row in summary.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) == 2:
                    label = cells[0].get_text(strip=True).rstrip(":")
                    value = cells[1].get_text(strip=True)
                    data[label] = value
        
        # Also grab the movie title from the page to confirm we got the right film
        h1 = soup.find("h1")
        data["page_title"] = h1.get_text(strip=True) if h1 else ""
        
        return data, url, r.status_code

    except Exception as e:
        return None, url, str(e)


# ── Main loop ─────────────────────────────────────────────────────────────────
results = []
failed = []

for i, row in oscars_modern.iterrows():
    film  = row['Film']
    year  = row['Year']
    slug  = row['slug']
    
    print(f"  [{i}] {film}...", end=" ")
    data, url, status = scrape_film(slug)
    
    if data:
        data['Film']      = film
        data['Year']      = year
        data['Ceremony']  = row['Ceremony']
        data['Winner']    = row['Winner']
        data['FilmId']    = row['FilmId']
        results.append(data)
        print(f"OK ({status})")
    else:
        failed.append({'Film': film, 'Year': year, 'slug': slug, 
                       'url': url, 'status': status})
        print(f"FAILED ({status})")
    
    time.sleep(2)  # polite pause

print(f"\nDone. Succeeded: {len(results)}, Failed: {len(failed)}")

  [8656] Before Night Falls... OK (200)
  [8657] Gladiator... FAILED (404)
  [8658] Cast Away... FAILED (404)
  [8659] Pollock... OK (200)
  [8660] Quills... OK (200)
  [8661] The Contender... FAILED (404)
  [8662] Shadow of the Vampire... OK (200)
  [8663] Traffic... OK (200)
  [8664] Erin Brockovich... OK (200)
  [8665] Gladiator... FAILED (404)
  [8666] The Contender... FAILED (404)
  [8667] Chocolat... FAILED (404)
  [8668] Requiem for a Dream... FAILED (404)
  [8669] You Can Count on Me... OK (200)
  [8670] Erin Brockovich... OK (200)
  [8671] Chocolat... FAILED (404)
  [8672] Pollock... OK (200)
  [8673] Almost Famous... OK (200)
  [8674] Almost Famous... OK (200)
  [8675] Billy Elliot... OK (200)
  [8676] Crouching Tiger, Hidden Dragon... OK (200)
  [8677] Dr. Seuss' How the Grinch Stole Christmas... FAILED (404)
  [8678] Gladiator... FAILED (404)
  [8679] Quills... OK (200)
  [8680] Vatel... OK (200)
  [8681] Crouching Tiger, Hidden Dragon... OK (200)
  [8682] Gladiator... FAIL

In [ ]:
# Clean up oscars.csv 
#bp['Winner'] = bp['Winner'].fillna(False)
#def make_slug(title):
 #   """Convert a film title to The Numbers URL slug."""
 #   slug = str(title).strip()
 #   slug = re.sub(r"[''']", "",  slug)       
 #   slug = re.sub(r"[:&,!?.]", "", slug)     
 #   slug = re.sub(r"\s+", "-", slug)         
 #   return slug

#bp['slug'] = bp['Film'].apply(make_slug)
#print(bp[['Film', 'slug']].head(20).to_string())
#bp_modern = bp[bp['Ceremony'] >= 73].copy()